In [ ]:
"""
tick_screen.py  (v2 — explainable)
==================================
Relevance screening for the OpenAlex tick corpus. Allow/deny lists are LOADED
AT RUNTIME from the screening_diagnostics CSVs (you never hand-maintain Python
lists). Every record is tagged with BOTH a tier and a human-readable reason.

WHY EACH RECORD GETS ITS TIER — the decision ladder (first match wins)
---------------------------------------------------------------------
  1. STRONG tick signal        -> "relevant"    (auto-keep; skip the LLM/BERTopic pass)
        - genus / family name in title / abstract / keywords      (near-unambiguous)
        - tick-specific MeSH descriptor (Ticks, Ixodes, ...)      (PubMed-curated)
        - tick concept / keyword vocab (Tick, Borrelia, ...)      (curated, high-precision)
        - PRIMARY topic in the tick allowlist                     (oa_topic only)
        - specialized tick / vector journal                       (repo-stripped)
  2. HARD deny                 -> "irrelevant"  (auto-drop)
        - field in denylist (Economics, Computer Science, Physics, ...)
        - domain == Social Sciences
        - PRIMARY topic in denylist (mite taxonomy, plant-insect)
  3. WEAK tick hint            -> "screen"      (send to BERTopic / multi-model LLM)
        - generic 'tick' or 'acari' in title / abstract
        - moderate vector / insect PRIMARY topic
        - broad parasitology subfield
  4. otherwise                 -> "irrelevant"  (no tick evidence anywhere in metadata)

CHANGES vs v1 (from the 2,000-row accuracy audit)
-------------------------------------------------
  FIX 1  topic allow/deny/moderate use the PRIMARY topic only (oa_topic), NOT
         oa_topics_all. v1's secondary-topic matching kept mosquito, mite and
         HIV papers because a low-scored secondary topic was tick-adjacent.
  FIX 2  subfield=Parasitology is demoted from auto-keep to WEAK evidence. As a
         strong signal it kept leeches / flukes (Parasitology = all parasites).
  FIX 3  simply being in a bio field is NO LONGER screen-worthy. A record with
         no genus/vocab/MeSH/allow-topic/journal AND no 'tick' word in title or
         abstract now drops to 'irrelevant' instead of bloating 'screen'.

Empirical basis (your 334k canonical set): 74% of records matched fulltext-only,
'acari' is the main false-positive (mites), and 'tick' in curated concepts is
high-precision while 'tick' in title/abstract is ambiguous.

Usage
-----
    from tick_screen import add_relevance, screening_report
    df = add_relevance(df, diag_dir="/content/.../screening_diag")   # loads CSVs
    # or  df = add_relevance(df)                                      # baked defaults
    print(screening_report(df))
    df[["oa_title","oa_relevance_tier","oa_relevance_reason"]].head(20)
"""

from __future__ import annotations
import os, re, glob
import pandas as pd

# ---- 1. mount + locate the file ----
from google.colab import drive
drive.mount('/content/drive')
pd.set_option('display.max_columns', None)

FILENAME = "master_canonical.parquet"          # or "combined.parquet"
hits = glob.glob(f"/content/drive/MyDrive/COGNOSX/ticks/preprocessing_data/{FILENAME}", recursive=True)
assert hits, (f"{FILENAME} not found under /content/drive. Is Drive synced? "
              f"Open the Files panel and confirm /content/drive is populated.")
PATH   = hits[0]
OUTDIR = os.path.join(os.path.dirname(PATH), "screening_diag")
print("Using :", PATH)
print("Output:", OUTDIR)

# ============================================================ regex logic ====
# genus + family names + hyphen variants of 'tick-borne' — near-unambiguous.
# NOTE: bare 'tick' and 'acari' are deliberately NOT here (ambiguous / mite-prone,
# handled as weak hints below).
GENERA = ["ixodes","amblyomma","rhipicephalus","boophilus","dermacentor","haemaphysalis",
          "hyalomma","ornithodoros","argas","otobius","carios","nuttalliella","rhipicentor",
          "margaropus","anomalohimalaya","cosmiomma","nosomma"]
FAMILIES = ["ixodidae","argasidae","ixodida","nuttalliellidae"]
SPECIFIC = GENERA + FAMILIES + ["tick-borne","tick borne","tickborne"]

_specific_re = re.compile(r"\b(?:" + "|".join(map(re.escape, SPECIFIC)) + r")\b", re.IGNORECASE)
_tickword_re = re.compile(r"\btick(?:s)?\b", re.IGNORECASE)     # ambiguous in title/abstract
_acari_re    = re.compile(r"\bacari\b", re.IGNORECASE)          # mite-prone -> weak only
# tick-specific MeSH (NOT bare 'acari' — that would catch mite descriptors)
_mesh_re     = re.compile(r"\btick|ixod|argas|amblyomma|rhipicephalus|dermacentor|hyalomma|"
                          r"haemaphysalis|ornithodoros|nuttalliell|arachnid vector", re.IGNORECASE)
# curated concept/keyword tick terms (high-precision even without title text)
_concept_re  = re.compile(r"\btick\b|ixodid|argasid|tick-borne|ixodes|nuttalliell", re.IGNORECASE)

# repositories / aggregators that appear as 'journal' but must never allowlist
REPO_STOP = ("all"), # ("pubmed","figshare","zenodo","biorxiv","medrxiv","research square","ssrn","doaj",
            #  "la referencia","cyberlenink","preprints","osf","semantic scholar","datacite",
            #  "dryad","mendeley","researchgate","hal ","repositor")
# fields we never deny (ticks / climate-tick papers live here)
PROTECT_FIELDS = {"environmental science", "medicine", "agricultural and biological sciences",
                  "veterinary", "biochemistry, genetics and molecular biology"}
# per-column minimum pos_rate to accept an ALLOW value when loading from CSV
ALLOW_MIN = {"oa_topic":0.60, "oa_subfield":0.40, "oa_field":0.40, "oa_journal":0.60,
             "oa_topics_all":0.50, "oa_mesh":0.65, "oa_concepts":0.60, "oa_keywords":0.60}

# ==================================================== baked-in fallbacks =====
# used only when a diag_*.csv is missing; the CSVs override these at runtime.
DEFAULTS = {
    "topic_allow": {"vector-borne infectious diseases","viral infections and vectors",
                    "vector-borne animal diseases","yersinia bacterium, plague, ectoparasites research"},
    "topic_deny": {"all"}, # {"study of mite species","insect-plant interactions and control"},
    "topic_moderate": {"mosquito-borne diseases and control","insect and pesticide research",
                       "insect pest control strategies","entomopathogenic microorganisms in pest control",
                       "bartonella species infections research","helminth infection and control",
                       "food allergy and anaphylaxis research","livestock and poultry management",
                       "vaccines and immunoinformatics approaches","invertebrate immune response mechanisms",
                       "species distribution and climate change"},
    "subfield_allow": {"parasitology","infectious diseases"},
    "field_allow": {"immunology and microbiology","veterinary"},
    "field_deny": {"all"}, # {"social sciences","neuroscience","engineering","arts and humanities","psychology",
                  #  "health professions","computer science","materials science",
                  #  "earth and planetary sciences","chemistry","economics, econometrics and finance",
                  #  "nursing","mathematics","business, management and accounting","energy",
                  #  "physics and astronomy","decision sciences","dentistry","chemical engineering"},
    "domain_deny": {"all"}, # {"social sciences"},
    "journal_allow": {"ticks and tick-borne diseases","vector-borne and zoonotic diseases",
                      "journal of medical entomology","medical and veterinary entomology",
                      "veterinary parasitology","journal of parasitology","parasitology research",
                      "parasitology international","experimental parasitology","international journal for parasitology",
                      "parasites & vectors","experimental and applied acarology","acta tropica",
                      "journal of vector ecology","american journal of tropical medicine and hygiene"},

    "vocab_allow": {"tick","ticks","ixodidae","ixodes","tick-borne","tick borne","tickborne","tick-borne disease",
                    "tick-borne encephalitis",
                    "rhipicephalus","amblyomma","hyalomma","haemaphysalis","dermacentor","argasidae",
                    "ixodida","nuttalliella","rhipicentor","boophilus","otobius",
                    "theileria","ornithodoros",
                    "margaropus","anomalohimalaya","cosmiomma","nosomma","nuttalliellidae"},
    "mesh_allow": {"tick","ticks","ixodes","ixodidae","tick-borne","tick borne","tickborne",
                   "tick infestations","tick-borne diseases","arachnid vectors",
                   "rhipicephalus","dermacentor","lyme disease","ornithodoros",
                   "nuttalliella","rhipicentor","boophilus",
                    "margaropus","anomalohimalaya","cosmiomma","nosomma","otobius","ixodida","nuttalliellidae"},
}

# ==================================================== list loading ==========
def load_lists(diag_dir: str | None) -> dict:
    """Read allow/deny sets from diag_*.csv, applying per-column thresholds,
    repo-stripping, and field/domain protection. Falls back to DEFAULTS per
    column whose CSV is absent."""
    L = {k: set(v) for k, v in DEFAULTS.items()}
    if not diag_dir or not os.path.isdir(diag_dir):
        return L

    def _load(col, kind="allow", n_min=20):
        p = os.path.join(diag_dir, f"diag_{col}.csv")
        if not os.path.exists(p):
            return None
        t = pd.read_csv(p, index_col=0)
        t.index = t.index.astype(str).str.strip().str.lower()
        if kind == "allow":
            sel = (t["pos_rate"] >= ALLOW_MIN.get(col, 0.5)) & (t["n_pos"] >= n_min)
        else:
            sel = (t["pos_rate"] <= 0.05) & (t["n_generic_only"] >= 20)
        return set(t[sel].index)

    if (s := _load("oa_topic", "allow")) is not None: L["topic_allow"] = s
    if (s := _load("oa_topic", "deny"))  is not None: L["topic_deny"]  = s | DEFAULTS["topic_deny"]
    if (s := _load("oa_subfield", "allow")) is not None: L["subfield_allow"] = s
    if (s := _load("oa_field", "allow")) is not None: L["field_allow"] = s
    if (s := _load("oa_field", "deny"))  is not None: L["field_deny"]  = s - PROTECT_FIELDS
    # domain is coarse: only 'social sciences' is safe to deny wholesale
    # (Physical Sciences contains Environmental/Earth Sciences -> climate-tick).
    if (s := _load("oa_domain", "deny")) is not None: L["domain_deny"] = s & {"social sciences"}
    if (s := _load("oa_journal", "allow")) is not None:
        L["journal_allow"] = {j for j in s if not any(r in j for r in REPO_STOP)}
    voc = set()
    for c in ("oa_concepts", "oa_keywords"):
        if (s := _load(c, "allow")) is not None: voc |= s
    if voc: L["vocab_allow"] = voc
    if (s := _load("oa_mesh", "allow")) is not None: L["mesh_allow"] = s
    L["field_deny"] -= PROTECT_FIELDS
    return L

# ==================================================== helpers ===============
def _txt(df, *cols):
    parts = [df[c].astype("string").fillna("") for c in cols if c in df.columns]
    if not parts:
        return pd.Series("", index=df.index)
    out = parts[0]
    for p in parts[1:]:
        out = out.str.cat(p, sep=" ")
    return out

def _lc(df, col):
    if col not in df.columns:
        return pd.Series("", index=df.index)
    return df[col].astype("string").fillna("").str.strip().str.lower()

def _multi_hit(df, col, allow_set):
    """True where any ' | '-split value of `col` is in allow_set (lowercased)."""
    if col not in df.columns or not allow_set:
        return pd.Series(False, index=df.index)
    return df[col].astype("string").fillna("").str.lower().apply(
        lambda s: bool(allow_set & {v.strip() for v in s.split("|")}))

# ==================================================== main ==================
def add_relevance(df: pd.DataFrame, diag_dir: str | None = None) -> pd.DataFrame:
    L = load_lists(diag_dir)
    df = df.copy()

    title_abs = _txt(df, "oa_title", "oa_abstract", "oa_keywords")
    title_kw = _txt(df, "oa_title", "oa_keywords")
    ticks_abs = _txt(df, "oa_abstract")
    mesh_txt  = _txt(df, "oa_mesh")
    topic    = _lc(df, "oa_topic")          # PRIMARY topic only (FIX 1)
    field    = _lc(df, "oa_field")
    domain   = _lc(df, "oa_domain")
    journal  = _lc(df, "oa_journal")
    subfield = _lc(df, "oa_subfield")

    # ---- STRONG positive signals (tick-specific or tight structural) --------
    f_genus   = title_abs.str.contains(_specific_re)                 # genus/family in text
    f_mesh    = mesh_txt.str.contains(_mesh_re) | _multi_hit(df, "oa_mesh", L["mesh_allow"])
    f_vocab   = (_txt(df, "oa_concepts", "oa_keywords").str.contains(_concept_re)
                 | _multi_hit(df, "oa_concepts", L["vocab_allow"])
                 | _multi_hit(df, "oa_keywords", L["vocab_allow"]))
    f_topic   = topic.isin(L["topic_allow"])                         # PRIMARY topic (FIX 1)
    f_journal = journal.isin(L["journal_allow"])
    f_tickword  = title_kw.str.contains(_tickword_re)               # generic 'tick' in text

    strong_pos = f_genus | f_mesh | f_vocab | f_topic | f_journal | f_tickword

    # ---- WEAK tick hints (route to 'screen', not auto-keep, not auto-drop) --
    f_tickabs  = ticks_abs.str.contains(_tickword_re)               # generic 'tick' in text
    f_acari     = title_abs.str.contains(_acari_re)                  # mite-prone term
    f_topic_mod = topic.isin(L["topic_moderate"])                    # PRIMARY moderate topic
    f_subf      = subfield.isin(L["subfield_allow"])                 # broad parasitology (FIX 2)
    weak = f_tickabs | f_acari | f_topic_mod | f_subf               # (no 'in a bio field') FIX 3

    # ---- HARD deny (structural homonym) -------------------------------------
    hard_deny = field.isin(L["field_deny"]) | domain.isin(L["domain_deny"]) | topic.isin(L["topic_deny"])

    # ---- decision ladder (first match wins) ---------------------------------
    tier = pd.Series("irrelevant", index=df.index)   # default: no evidence -> drop
    tier[weak] = "screen"                            # weak hint -> screen
    tier[hard_deny] = "irrelevant"                   # structural homonym overrides a weak hint
    tier[strong_pos] = "relevant"                    # strong signal wins outright
    df["oa_relevance_tier"] = tier

    # ---- explainability: one reason per record ------------------------------
    reason = pd.Series("", index=df.index)
    R = tier.eq("relevant")
    reason[R & f_genus & reason.eq("")]   = "genus/family name in title/abstract"
    reason[R & f_mesh & reason.eq("")]    = "tick-specific MeSH term"
    reason[R & f_vocab & reason.eq("")]   = "tick concept/keyword vocab"
    reason[R & f_topic & reason.eq("")]   = "primary topic in tick allowlist"
    reason[R & f_journal & reason.eq("")] = "specialized tick/vector journal"
    reason[R & f_tickword & reason.eq("")]  = "generic 'tick' in title/keywords"
    S = tier.eq("screen")
    reason[S & f_tickabs & reason.eq("")]  = "generic 'tick' in abstract (ambiguous)"
    reason[S & f_acari & reason.eq("")]     = "acari/mite term (possible mite, not tick)"
    reason[S & f_topic_mod & reason.eq("")] = "moderate vector/insect topic"
    reason[S & f_subf & reason.eq("")]      = "broad parasitology subfield"
    I = tier.eq("irrelevant")
    reason[I & hard_deny & reason.eq("")] = "off-topic field/domain/topic (homonym)"
    reason[I & reason.eq("")]             = "no tick evidence in metadata"
    df["oa_relevance_reason"] = reason

    # audit flags (which strong signal fired)
    df["rel_genus"], df["rel_mesh"], df["rel_vocab"] = f_genus, f_mesh, f_vocab
    df["rel_topic"], df["rel_journal"], df["rel_deny"] = f_topic, f_journal, hard_deny
    return df

# ==================================================== reporting =============
def screening_report(df: pd.DataFrame) -> str:
    vc = df["oa_relevance_tier"].value_counts(); tot = len(df)
    lines = ["RELEVANCE TIERS (v2, explainable)"]
    for t in ("relevant", "screen", "irrelevant"):
        c = int(vc.get(t, 0))
        lines.append(f"  {t:11s}: {c:>8,}  ({c/tot:.1%})")
    lines.append("  --- reasons ---")
    for reason, c in df["oa_relevance_reason"].value_counts().items():
        lines.append(f"    {c:>8,}  {reason}")
    return "\n".join(lines)

In [ ]:
# from tick_screen import add_relevance, screening_report

df = pd.read_parquet(PATH)
df = add_relevance(df, diag_dir=OUTDIR)     # OUTDIR = your screening_diag folder
print(screening_report(df))
df.to_parquet(os.path.join(os.path.dirname(PATH), "canonical_screened.parquet"), index=False)
df.to_csv(os.path.join(os.path.dirname(PATH), "canonical_screened.csv"), index=False)

In [ ]:
# from tick_screen import add_relevance, screening_report

df = pd.read_parquet(PATH)
df = add_relevance(df, diag_dir=OUTDIR)     # OUTDIR = your screening_diag folder
print(screening_report(df))
df.to_parquet(os.path.join(os.path.dirname(PATH), "canonical_screened.parquet"), index=False)
df.to_csv(os.path.join(os.path.dirname(PATH), "canonical_screened.csv"), index=False)

In [ ]:
# from tick_screen import add_relevance, screening_report

df = pd.read_parquet(PATH)
df = add_relevance(df, diag_dir=OUTDIR)     # OUTDIR = your screening_diag folder
print(screening_report(df))
df.to_parquet(os.path.join(os.path.dirname(PATH), "canonical_screened.parquet"), index=False)
df.to_csv(os.path.join(os.path.dirname(PATH), "canonical_screened.csv"), index=False)

In [ ]:
df